# figure8.ipynb

Independently exports Figure 8 (Labrador Sea) from the final combined figure8.ipynb source.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

In [ ]:
fdir1 = configured_path('reccap_mask_file')
mask = xr.open_mfdataset(fdir1)
mask_ocean = mask.open_ocean

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-01',
    periods=ds.sizes['time'],
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)
ds = ds.where(mask_ocean)
ds

In [ ]:
fdir2 = configured_path('iap_mld_file')
ds2 = xr.open_mfdataset(fdir2)

time_new = pd.date_range(
    start='1940-01-1',
    end='2023-12-15',
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds2 = ds2.assign_coords(time=time_new)

ds2 = ds2.sel(time=slice('1993','2023'))
ds2 = ds2.where(mask_ocean)
ds2

In [ ]:
fdir3 = configured_path('oras_mld_pattern')
ds3 = xr.open_mfdataset(fdir3)
ds3 = ds3.resample(time='1ME').mean(dim='time')

time_new = pd.date_range(
    start='1993-01-1',
    end='2023-12-15',
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds3 = ds3.assign_coords(time=time_new)
ds3 = ds3.where(mask_ocean)
ds3

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from pathlib import Path
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmaps
import matplotlib.ticker as ticker
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Rectangle
import matplotlib.path as mpath
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter

times_font_dir = Path(configured_path('font_dir'))
times_font_files = [times_font_dir / 'Times New Roman.ttf', times_font_dir / 'Times New Roman Bold.ttf']
for font_file in times_font_files:
    if not font_file.is_file():
        raise FileNotFoundError(f'Required Times New Roman font not found: {font_file}')
    font_manager.fontManager.addfont(str(font_file))
resolved_times = Path(font_manager.findfont('Times New Roman', fallback_to_default=False)).resolve()
if resolved_times not in {font.resolve() for font in times_font_files}:
    raise RuntimeError(f'Times New Roman resolved to an unexpected file: {resolved_times}')
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'axes.unicode_minus': False,
})

# 自定义色板
cmap1 = cmaps.precip_11lev[1:]
cmap2 = cmap1.interp(20)

# ====================== 1. 计算DJF冬季多年气候态（跨年12/1/2） ======================
def calc_djf_clim(ds_in):
    ds_sub = ds_in.sel(time=slice("1993-1", "2023-8"))
    ds_djf = ds_sub.where(ds_sub.time.dt.season == "DJF", drop=True)
    djf_year = ds_djf.time.dt.year.where(ds_djf.time.dt.month != 12, ds_djf.time.dt.year + 1)
    ds_djf = ds_djf.assign_coords(djf_year=djf_year)
    ds_djf_year = ds_djf.groupby("djf_year").mean("time", skipna=True)
    ds_djf_clim = ds_djf_year.mean("djf_year", skipna=True)
    return ds_djf_clim

# 数据集分别计算DJF气候态
ds_region = calc_djf_clim(ds)
ds2_region = calc_djf_clim(ds2.mld)
ds3_region = calc_djf_clim(ds3.MLD_sigma0_030)

# ====================== 2. 提取DJF多年平均MLD场 ======================
mld_pred = ds_region["MLD_pred"].compute()
mld_true = ds_region["MLD_true"].compute()
mld_iap  = ds2_region.compute()
mld_oras = ds3_region.compute()

plot_info = [
    (mld_pred, "SJTU-ViT"),
    (mld_true, "Obs"),
    (mld_iap, "IAP"),
    (mld_oras, "ORAS5")
]

# 与 figure9 一致的北大西洋区域平均时间序列（2010–2020）
ts_ds = ds.sel(lat=slice(56, 60), lon=slice(304, 309), time=slice('2010', '2020'))
ts_iap = ds2.mld.sel(lat=slice(56, 60), lon=slice(304, 309), time=slice('2010', '2020'))
ts_oras = ds3.MLD_sigma0_030.sel(lat=slice(56, 60), lon=slice(304, 309), time=slice('2010', '2020'))
ts_series = [
    ("SJTU-ViT", ts_ds.MLD_pred.mean(["lat", "lon"], skipna=True).compute(), "#d62728"),
    ("Obs", ts_ds.MLD_true.mean(["lat", "lon"], skipna=True).compute(), "#1f77b4"),
    ("IAP", ts_iap.mean(["lat", "lon"], skipna=True).compute(), "#17becf"),
    ("ORAS5", ts_oras.mean(["lat", "lon"], skipna=True).compute(), "#2ca02c"),
]

# ====================== 非线性色阶 ======================
nonlinear_levels = [10, 20, 50, 100, 150, 220, 300, 400, 550, 800]
norm = BoundaryNorm(nonlinear_levels, ncolors=cmap2.N, extend="max")

# ====================== 投影 ======================
# 比 LambertConformal 更接近你给的“扇形极区图”效果
proj = ccrs.NorthPolarStereo(central_longitude=-37.5)

# 地图范围
lon_min, lon_max = -70, -5
lat_min, lat_max = 52, 68

# 红色框研究区：56–60°N, 51~56°W
target_lon_min, target_lon_max = -56, -51
target_lat_min, target_lat_max = 56, 60

def build_boundary_path(projection, lon_min, lon_max, lat_min, lat_max, n=300):
    """
    沿经纬度矩形边界采样，并投影到当前投影坐标系，
    用于构造扇形/弧形外框。
    """
    # 下边
    lons1 = np.linspace(lon_min, lon_max, n)
    lats1 = np.full(n, lat_min)

    # 右边
    lons2 = np.full(n, lon_max)
    lats2 = np.linspace(lat_min, lat_max, n)

    # 上边
    lons3 = np.linspace(lon_max, lon_min, n)
    lats3 = np.full(n, lat_max)

    # 左边
    lons4 = np.full(n, lon_min)
    lats4 = np.linspace(lat_max, lat_min, n)

    lons = np.concatenate([lons1, lons2, lons3, lons4])
    lats = np.concatenate([lats1, lats2, lats3, lats4])

    pts = projection.transform_points(ccrs.PlateCarree(), lons, lats)
    verts = pts[:, :2]

    verts = np.vstack([verts, verts[0]])
    codes = np.full(len(verts), mpath.Path.LINETO, dtype=np.uint8)
    codes[0] = mpath.Path.MOVETO
    codes[-1] = mpath.Path.CLOSEPOLY

    return mpath.Path(verts, codes)

boundary_path = build_boundary_path(proj, lon_min, lon_max, lat_min, lat_max, n=400)

# ====================== 3. 2×2地图 + 底部时间序列 ======================
fig = plt.figure(figsize=(9.0, 7.8), dpi=300)
gs = fig.add_gridspec(
    3, 3, width_ratios=[1, 1, 0.065], height_ratios=[1, 1, 0.95],
    left=0.06, right=0.95, bottom=0.08, top=0.96, wspace=0.04, hspace=0.13
)
axes_flat = [
    fig.add_subplot(gs[0, 0], projection=proj),
    fig.add_subplot(gs[0, 1], projection=proj),
    fig.add_subplot(gs[1, 0], projection=proj),
    fig.add_subplot(gs[1, 1], projection=proj),
]

for idx, ((data, title), ax) in enumerate(zip(plot_info, axes_flat)):
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    # 关键：真正裁成扇形边界
    ax.set_boundary(boundary_path, transform=ax.transData)

    # 去掉默认矩形边框
    for s in ["left", "right", "top", "bottom"]:
        ax.spines[s].set_visible(False)

    # 填色等值线
    cs = ax.contourf(
        data.lon, data.lat, data,
        levels=nonlinear_levels,
        norm=norm,
        cmap=cmap2,
        transform=ccrs.PlateCarree(),
        extend="max"
    )

    # 陆地、海岸线
    ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=3)
    ax.add_feature(cfeature.COASTLINE, lw=0.5, zorder=4)

    # 经纬度网格
    gl = ax.gridlines(draw_labels=False, lw=0.3, alpha=0.7, color="black")
    gl.ylocator = ticker.MultipleLocator(4)
    gl.xlocator = ticker.MultipleLocator(10)
    
    # 不让标签沿着网格线旋转，保证经度标签横过来
    gl.rotate_labels = False
    
    # 让标签不要贴在网格线上
    gl.x_inline = False
    gl.y_inline = False
    
    # 经度、纬度格式
    gl.xformatter = LongitudeFormatter(zero_direction_label=True)
    gl.yformatter = LatitudeFormatter()
    
    # 经度标签横向显示
    gl.xlabel_style = {
        "size": 9,
        "rotation": 0,
        "ha": "center",
        "va": "top",
    }    
    # 纬度标签放在图外一点
    gl.ylabel_style = {
        "size": 9,
        "rotation": 0,
        "ha": "right",
        "va": "center",
    }
    
    # 标签往外挪一点
    gl.xpadding = 6
    gl.ypadding = 8

    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = False
    gl.bottom_labels = False

    # 红色虚线框
    box = Rectangle(
        (target_lon_min, target_lat_min),
        target_lon_max - target_lon_min,
        target_lat_max - target_lat_min,
        transform=ccrs.PlateCarree(),
        fill=False,
        edgecolor="crimson",
        lw=1.1,
        linestyle="--",
        zorder=5
    )
    ax.add_patch(box)

    ax.set_title(title, fontsize=15, loc="center", pad=2)
    ax.text(0.0, 1.02, f"({chr(97 + idx)})", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=13)

# 地图统一色条
cbar_ax = fig.add_subplot(gs[:2, 2])
cbar = fig.colorbar(
    cs,
    cax=cbar_ax,
    ticks=nonlinear_levels,
    format=ticker.ScalarFormatter(),
    spacing="uniform"
)
cbar.set_label("Climatological DJF MLD (m)", fontsize=13)
cbar.ax.tick_params(labelsize=12)

# 底部区域平均时间序列
ax_ts = fig.add_subplot(gs[2, :])
for label, series, color in ts_series:
    ax_ts.plot(series.time.values, series.values, color=color, lw=1.5, label=label)
ax_ts.legend(loc="upper left", ncol=1, frameon=True, fontsize=10)
ax_ts.set_title("Regional Averaged MLD (56–60°N, 304–309°E)", fontsize=13, pad=4)
ax_ts.set_xlabel("Date (YYYY)", fontsize=12)
ax_ts.set_ylabel("MLD (m)", fontsize=12)
ax_ts.set_ylim(0, 2000)
ax_ts.xaxis.set_major_locator(mdates.YearLocator(2))
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax_ts.grid(alpha=0.3, linewidth=0.4)
ax_ts.tick_params(axis="both", labelsize=10.5)
ax_ts.text(0.0, 1.04, "(e)", transform=ax_ts.transAxes,
           ha="left", va="bottom", fontsize=13)

output_dir = Path(configured_path('output_dir'))
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / 'figure8.png',
            dpi=600, bbox_inches='tight', pad_inches=0)
plt.show()
plt.close("all")